In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
import geopandas as gpd
import scipy.stats as stats
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import rioxarray as rxt
import fiona
import scipy.stats as stats

%matplotlib qt

In [ ]:
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

# 임상도 불러오기

In [ ]:
# function: get shape and transform information from reference raster (=DEM raster)
def getReferenceRasterInfo (ref_epsg, raster_name):
    ref_epsg = 5179
    print('Reading raster file...')
    with rasterio.open(raster_name) as src:
        dem_meta = src.meta.copy()
        dem_crs = src.crs
        dem_transform = src.transform
        dem_shape = (src.height, src.width)
        print('Read information from DEM raster')
    
        # Reproject DEM
        if int(dem_crs.to_wkt()[-7:-3]) != ref_epsg:
            print('Start Reprojection of DEM...')
            dst_crs = f'EPSG:{ref_epsg}'
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, src.width, src.height, *src.bounds
            )
            new_meta = src.meta.copy()
            new_meta.update({
            'crs' : dst_crs,
            'transform' : transform,
            'width' : width,
            'height' : height
        })
        
            reproejct_raster_name = r"F:\CBH\DEM_merge_5179_Reproject.tif"
            with rasterio.open(reproejct_raster_name, 'w', **new_meta) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination = rasterio.band(dst, 1),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.bilinear
                )
            print('Reprojeciton of DEM Ends...')
            dem_info = {'meta' : new_meta, 'shape' : dem_shape, 'transform' : dem_transform, 'crs' : dst_crs}
        else: 
            ('Skip Raster Reprojection...')
            dem_info = {'meta' : dem_meta, 'shape' : dem_shape, 'transform' : dem_transform, 'crs' : dem_crs}
    return dem_info
    

def rasterizeAttribute (gdb_dir, layer_name, target_attribute_lst, dem_meta):
    layers = fiona.listlayers(gdb_dir)
    if layer_name in layers:
        imsang = gpd.read_file(os.path.join(gdb_dir), layer=layer_name)
    else: (f'{layer_name} does not exist in {gdb_dir}')
    
    # if crs doesn't match, reproject shpaefile
    ref_epsg = dem_info['crs'].to_wkt()[-7:-3]
    if (imsang.crs.to_epsg()) != ref_epsg:
        imsang = imsang.to_crs(ref_epsg)
    else: print('Skip reprojection...')

    # string type code to integer type
    dbh_dict = {'0' : 0, '1' : 1, '2' : 2, '3': 3}
    h_dict = {f"{i*2:02d}" : i*2 for i in range(21)} # f':02d' : 10 미만의 수는 0으로 padding 넣기
    cd_dict = {'A' : 1, 'B' : 2, 'C' : 3}
    all_dict = {'DMCLS_CD' : dbh_dict, 'HEIGHT' : h_dict, 'DNST_CD' : cd_dict}
    # Copy meta and update
    meta = dem_meta.copy()
    meta.update({'count': 1, 'dtype': rasterio.float32, 'nodata': -99})
    for target_att in tqdm(target_attribute_lst, desc='Rasterize...', position=0):

        if target_att in all_dict.keys():
            selected_dict = all_dict[target_att]
            imsang.loc[:,target_att] = imsang.loc[:,target_att].apply(lambda x : selected_dict[x] if x in selected_dict.keys() else -99)
        elif target_att == 'KOFTR_GROU':
            imsang.loc[:,target_att] = imsang.loc[:,target_att].astype('int')
        shapes = [(geom, v1) for geom, v1 in zip(imsang.geometry, imsang[target_att])] # DBH

        rasterized_imsang = rasterio.features.rasterize(
            shapes = shapes,
            out_shape = (meta['height'], meta['width']),
            transform = dem_info['transform'],
            fill=-99,
            dtype = rasterio.float32
        )
        
        print('Rasterization of imsang shapefile Ends')
        
        result_dir = r"D:/ForestFire/CBH/result"
        # save the rasterized imsang shape
        imsang_raster = os.path.join(result_dir, f'{feature_list[0]}_raster.tif')

        if meta.get('width') is None or meta.get('height') is None:
            raise ValueError("Width and height must not be None")
        with rasterio.open(imsang_raster, 'w', **meta) as dst:
            dst.write(rasterized_imsang, 1)
        print('Save imsang raster file')
        
    return imsang_raster





# raster화된 속성별 임상도에 random sampling으로 값 부여
class attributeImputer:
    def __init__ (self):
        self.sample_dict = {'dbh' : None, 'height' : None, 'crown density' : None}


    def rasterIterator(self, input_raster, output_raster, target_feature, grid_size=[1, 1]):
        def randomSampling(self, patch_data, target_feature):
            # Extract value from distribution
            dbh_dict = {0 : [0, 6], 1 : [6, 18], 2 : [18, 30], 3 : [30, 107]}
            h_dict = {i*2: [i*2 - 1, i*2 + 1]  if (i > 0) else [0, 1] for i in range(21)}
            h_dict[42] = [41, 50] # 단위; m # f':02d' : 10 미만의 수는 0으로 padding 넣기
            cd_dict = {1 : [0, 50], 2 : [51, 70], 3 : [71, 100]}
            # dictionary of feature-dictionary:
            feature_dict = {'dbh' : dbh_dict, 'height' : h_dict, 'crown density' : cd_dict}
            selected_dict = feature_dict[target_feature]
    
            # filter data and fill with samples
            band_cnt, height, width = np.shape(patch_data)
            flatten = patch_data.flatten()
            imputed = np.full(flatten.shape, -99., dtype=np.float32)
            for key in tqdm(selected_dict.keys(), desc='imputing data', leave=False, position=1):
                lower, upper = selected_dict[key]
                samples = self.sample_dict[target_feature]
                mask = flatten == key
                
                if np.any(mask):
                    filtered_sample = samples[(samples > lower) & (samples <= upper) & (samples > 0.)]
                    # Hierarchical fallback
                    if len(filtered_sample) == 0:
                        print(f"[Warning] No sample for key {key}. Trying lower class...")
                        lower_key = key - 2 if key - 2 in selected_dict else None
                        if lower_key is not None:
                            lower_low, upper_low = selected_dict[lower_key]
                            filtered_sample = samples[(samples > lower_low) & (samples <= upper_low) & (samples > 0.)]
                            if len(filtered_sample) > 0:
                                print(f" → Fallback: Using class {lower_key} samples.")
                        # Final fallback
                        if len(filtered_sample) == 0:
                            print(f" → No lower class sample. Using global samples.")
                            filtered_sample = samples[samples > 0.]
        
                    sampled_values = np.random.choice(filtered_sample, size=np.sum(mask), replace=True)
                    imputed[mask] = sampled_values
                    
            return imputed.reshape((band_cnt, height, width))
                
            
        with rasterio.open(input_raster) as src:
            print('Reading input raster...')
            new_meta = src.meta.copy()
            width, height = src.width, src.height

            # prepare variables for the processing
            processed_data = np.full((src.count, height, width), fill_value = src.nodata, dtype=rasterio.float32)
            patch_height = height // grid_size[0]
            patch_width = width // grid_size[1]
    
            # iterate raster using window
            for i in tqdm(range(grid_size[0]), desc = 'processing patches...', leave=True, position=0): # height
                for j in range(grid_size[1]): # width
                    window = rasterio.windows.Window(patch_width * j, patch_height * i, patch_width, patch_height)
                    patch_data = src.read(window=window)
                    processed_patch = randomSampling(self, patch_data, target_feature)
                    # merge patches
                    processed_data[:, i*patch_height : (i+1)*patch_height, j*patch_width : (j+1)*patch_width] = processed_patch
        new_meta.update({'dtype' : rasterio.float32})
        # save processed raster into new raster
        with rasterio.open(output_raster, 'w', **new_meta) as dst:
            print('Saving imputed raster...')
            dst.write(processed_data)
        print('Process Ends!')
        return output_raster
    
    def createSamples (self, data, attribute, size, bound = None):
        def find_best_fit_distribution(self, filtered, distributions):
            results = {}
            x = np.linspace(filtered.min(), filtered.max(), 1000)
        
            for name, dist in distributions.items():
                try:
                    # Fit distribution to data
                    params = dist.fit(filtered)
                    pdf_fitted = dist.pdf(x, *params)
                    ks_stat, ks_pval = stats.kstest(filtered, lambda x: dist.cdf(x, *params))
        
                    # Store results
                    results[name] = {
                        "params": params,
                        "KS Statistic": ks_stat,
                        "p-value": ks_pval,
                        "pdf": pdf_fitted
                    }
                except Exception as e:
                    print(f"Skipping {name} due to error: {e}")
        
            # Select best fit (highest p-value)
            best_fit = max(results, key=lambda d: results[d]["p-value"])
            return best_fit, results
            
        distributions = {
        "Gamma": stats.gamma,
        "Log-Normal": stats.lognorm,
        "Beta": stats.beta,
        "Weibull": stats.weibull_min,
        "Exponential": stats.expon,
        "GEV" : stats.genextreme
        } 
    
        if bound is not None:
            lower = bound[0] if bound[0] is not None else -np.inf
            upper = bound[1] if bound[1] is not None else np.inf
            condition = (data[attribute] > lower) & (data[attribute] <= upper)
            filtered_data = data.loc[condition, attribute].dropna()
        else:
            lower = -np.inf
            upper = np.inf
            filtered_data = data[attribute].dropna()
            
        best_fit, results = find_best_fit_distribution(self, filtered_data, distributions)
        best_dist = distributions[best_fit]
        print(best_fit)
        
        samples = []
        while len(samples) < size:
            new_sample = best_dist.rvs(*results[best_fit]['params'], size=size, random_state=44)
            filtered_sample = new_sample[(new_sample > lower) & (new_sample <= upper)]
            samples.extend(filtered_sample.tolist())
        samples = np.array(samples[:size])
        print(f'Created sample size: {len(samples)}')

        # plot the result of best_fit
        x = np.linspace(filtered_data.min(), filtered_data.max(), 1000)
        pdf_fitted = best_dist.pdf(x, *results[best_fit]['params'])
        
        plt.figure(figsize=(8, 5))
        plt.hist(filtered_data, bins=30, density=True, color='gray', alpha=0.6, label="Data Histogram")
        plt.plot(x, pdf_fitted, label=f"Best Fit: {best_fit}", linewidth=2, color="red")
        plt.xlabel(f"{attribute}")
        plt.ylabel("Probability Density")
        plt.title(f"Best-Fitting Probability Distribution for {attribute}")
        plt.legend()
        plt.show()
        
        # Print best fit results
        print(f"Best-Fitting Distribution: {best_fit}")
        print(f"Parameters: {results[best_fit]['params']}")
        print(f"KS Statistic: {results[best_fit]['KS Statistic']}")
        print(f"P-Value: {results[best_fit]['p-value']}")
    
        # save samples to the class variable
        if attribute == '흉고직경' : self.sample_dict['dbh'] = samples
        elif attribute == '수고' : self.sample_dict['height'] = samples
        elif attribute == '평균수관밀도(%)' : self.sample_dict['crown density'] = samples
        else: self.sample_dict[attribute] = samples
            
        return samples, best_fit, results

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from tqdm import tqdm


def createCBHRaster(params_file, attribute_map_list, output_header, grid_height, grid_width, verbose=True):
    """
    Create CBH and CR raster based on attribute rasters and species-specific parameters.

    :param params_file: Path to CSV file containing species parameters
    :param attribute_map_list: List of raster paths [species, dbh, height, cr, dem, slope, azimuth]
    :param output_header: Output folder path
    :param grid_height: Number of grid rows for patch processing
    :param grid_width: Number of grid columns for patch processing
    :param verbose: Whether to show progress bars
    """

    def calculateCBH(patch_data, params_dict, nodata):
        def func2(H, D, EL, SL, AZ, CD, a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6):
            H_log, D_log = np.log1p(H), np.log1p(D)
            size = (b1 * H_log / D_log) + (b2 * H_log) + (b3 * D_log ** 2)
            comp = c1 * CD
            site = (d1 * EL) + (d2 * EL ** 2) + (d3 * SL) + (d4 * SL ** 2) + (d5 * SL * np.sin(AZ)) + (d6 * SL * np.cos(AZ))
            x = size + comp + site + a
            cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
            return cr

        patch_species, patch_dbh, patch_h, patch_cd, patch_dem, patch_slope, patch_azimuth = patch_data

        if patch_species.shape != patch_dem.shape:
            raise ValueError("The shapes of the rasters don't match")

        band_cnt, tile_height, tile_width = patch_dem.shape
        species_flat = patch_species.flatten()
        dbh_flat = patch_dbh.flatten()
        height_flat = patch_h.flatten()
        cd_flat = patch_cd.flatten()
        dem_flat = patch_dem.flatten()
        slope_flat = patch_slope.flatten()
        azimuth_flat = patch_azimuth.flatten()

        valid_mask = (
            (species_flat != nodata[1]) &
            (dem_flat != nodata[0]) &
            (slope_flat != nodata[0]) &
            (azimuth_flat != nodata[0]) &
            (azimuth_flat != -1.)
        )

        patch_cr = np.full(dem_flat.shape, np.nan, dtype=np.float32)
        patch_cbh = np.full(dem_flat.shape, np.nan, dtype=np.float32)

        species_ids = np.unique(species_flat[valid_mask])
        for sid in species_ids:
            code_mask = (species_flat == sid) & valid_mask
            if not np.any(code_mask):
                continue

            height = height_flat[code_mask] * 3.28084  # m to ft
            dbh = dbh_flat[code_mask] * 0.3937  # cm to inch
            cd = cd_flat[code_mask]
            dem = dem_flat[code_mask] / 100  # m to hm
            slope = slope_flat[code_mask]
            az = np.radians(azimuth_flat[code_mask])

            if sid not in params_dict:
                continue

            params = params_dict[sid].values
            cr_vals = func2(height, dbh, dem, slope, az, cd, *params)
            cbh_vals = (1 - cr_vals) * height / 3.28084 # ft to m

            patch_cr[code_mask] = cr_vals
            patch_cbh[code_mask] = cbh_vals

        return (
            patch_cr.reshape((band_cnt, tile_height, tile_width)),
            patch_cbh.reshape((band_cnt, tile_height, tile_width))
        )

    # === Main process ===
    print('[INFO] Preparing species parameters...')
    params = pd.read_csv(params_file, encoding='cp949')
    params_df = params[['SID'] + [f'Par{i}' for i in range(11)]]
    params_dict = {int(row['SID']): row.iloc[1:] for _, row in params_df.iterrows()}

    # === Open rasters ===
    with rasterio.open(attribute_map_list[0]) as species_ras, \
         rasterio.open(attribute_map_list[1]) as dbh_ras, \
         rasterio.open(attribute_map_list[2]) as height_ras, \
         rasterio.open(attribute_map_list[3]) as cr_ras, \
         rasterio.open(attribute_map_list[4]) as dem_ras, \
         rasterio.open(attribute_map_list[5]) as slope_ras, \
         rasterio.open(attribute_map_list[6]) as azimuth_ras:

        print('[INFO] Reading input rasters...')
        meta = dem_ras.meta.copy()
        width, height = dem_ras.width, dem_ras.height

        cbh_data = np.full((1, height, width), fill_value=dem_ras.nodata, dtype=np.float32)
        cr_data = np.full((1, height, width), fill_value=dem_ras.nodata, dtype=np.float32)

        patch_tile_height = height // grid_height
        patch_tile_width = width // grid_width

        for i in tqdm(range(grid_height), desc='Row patches', leave=True, position=0):
            for j in tqdm(range(grid_width), desc='Col patches', leave=False, position=1):# , disable=not verbose
                window = Window(patch_tile_width * j, patch_tile_height * i, patch_tile_width, patch_tile_height)
                patch_species = species_ras.read(window=window)
                patch_dbh = dbh_ras.read(window=window)
                patch_h = height_ras.read(window=window)
                patch_cd = cr_ras.read(window=window)
                patch_dem = dem_ras.read(window=window)
                patch_slope = slope_ras.read(window=window)
                patch_azimuth = azimuth_ras.read(window=window)

                patch_data = [patch_species, patch_dbh, patch_h, patch_cd, patch_dem, patch_slope, patch_azimuth]
                dem_nodata = dem_ras.nodata
                species_nodata = species_ras.nodata

                processed_cr, processed_cbh = calculateCBH(patch_data, params_dict, [dem_nodata, species_nodata])

                cbh_data[:, i * patch_tile_height: (i + 1) * patch_tile_height, j * patch_tile_width: (j + 1) * patch_tile_width] = processed_cbh
                cr_data[:, i * patch_tile_height: (i + 1) * patch_tile_height, j * patch_tile_width: (j + 1) * patch_tile_width] = processed_cr

    # === Save rasters ===
    meta.update({'dtype': np.float32})
    os.makedirs(output_header, exist_ok=True)

    cbh_path = os.path.join(output_header, 'CBH.tif')
    cr_path = os.path.join(output_header, 'CR.tif')

    with rasterio.open(cbh_path, 'w', **meta) as dst:
        dst.write(cbh_data)
        print(f'[Message] CBH raster saved: {cbh_path}')

    with rasterio.open(cr_path, 'w', **meta) as dst:
        dst.write(cr_data)
        print(f'[Message] CR raster saved: {cr_path}')

    print('[Message] Process complete!')
    return cbh_path, cr_path


# TEST - raster imputeer

In [ ]:
# attributeImputer test
parent_dir = r"D:/ForestFire/CBH"
df = pd.read_csv(os.path.join(parent_dir, r'data/NFI7-Immok-Filtered3.csv'))
print(df.columns)
imputer = attributeImputer()
samples, best_fit, results = imputer.createSamples(df, '흉고직경', 150000, [0, None])
input_raster = r"D:\ForestFire\CBH\result\DMCLS_CD_raster.tif"
output_raster = r"D:\ForestFire\CBH\result\DMCLS_CD_sampled.tif"
imputer.rasterIterator(input_raster, output_raster, target_feature = 'dbh', grid_size=[5, 5]) # traget_feature: 'dbh','height','crown density'

In [ ]:
# 수종 basemap 만들기
feature_list = ['HEIGHT'] # 'DMCLS_CD', # 'KOFTR_GROU'
dem_info = getReferenceRasterInfo("5179", r"F:\CBH\DEM_merge_5179.tif")

In [ ]:
imsang = rasterizeAttribute(r"G:\CBH\Imsang_merge.gdb", "merge", feature_list, dem_info['meta'])

In [ ]:
# NFI 자료 DF으로 불러오기
parent_dir = r"D:/ForestFire/CBH"
df = pd.read_csv(os.path.join(parent_dir, r'data/NFI7-Immok-Filtered3.csv'))
df['수고'] = df['수고'].apply(lambda x: x * 0.01) # m로 단위 변환
print(df.columns)

In [ ]:
# Imputer 선언
imputer = attributeImputer()

# 수고레스터 만들기
samples, best_fit, results = imputer.createSamples(df, '수고', 150000, [0, (max(df['수고']) + 2)])
input_raster = r"D:\ForestFire\CBH\result\HEIGHT_raster.tif"
output_raster = r"D:\ForestFire\CBH\result\HEIGHT_sampled.tif"
imputer.rasterIterator(input_raster, output_raster, target_feature = 'height', grid_size=[5, 5]) # traget_feature: 'dbh','height','crown density'

# 흉고직경레스터
samples, best_fit, results = imputer.createSamples(df, '흉고직경', 150000, [0, None])
input_raster = r"D:\ForestFire\CBH\result\DMCLS_CD_raster.tif"
output_raster = r"D:\ForestFire\CBH\result\DMCLS_CD_sampled.tif"
imputer.rasterIterator(input_raster, output_raster, target_feature = 'dbh', grid_size=[5, 5]) # traget_feature: 'dbh','height','crown density'

# 평균수관밀도레스터
samples, best_fit, results = imputer.createSamples(df, '평균수관밀도(%)', 150000, [0, 100])
input_raster = r"D:\ForestFire\CBH\result\DNST_CD_raster.tif"
output_raster = r"D:\ForestFire\CBH\result\DNST_CD_sampled.tif"
imputer.rasterIterator(input_raster, output_raster, target_feature = 'crown density', grid_size=[5, 5]) # traget_feature: 'dbh','height','crown density'

In [ ]:
with rasterio.open(r"D:\ForestFire\CBH\result\HEIGHT_sampled.tif") as src1, \
rasterio.open(r"D:\ForestFire\CBH\result\DMCLS_CD_sampled.tif") as src2, \
rasterio.open(r"D:\ForestFire\CBH\result\DNST_CD_sampled.tif") as src3:
    heigth_bnd = src1.read(1)
    dbh_bnd = src2.read(1)
    density_bnd = src3.read(1)

In [ ]:
index = np.random.randint(0, cleaned2.shape[0], 1000000)
sampled = cleaned2[index]
sampled

In [ ]:
def drawPlot (bnd, attribute):
    cleaned = bnd[(~np.isnan(bnd)) & (bnd != -99.)]
    plt.figure(figsize=(8, 5))
    plt.hist(df[attribute], bins=30, density=True, color='blue', alpha=0.2, label="Observed")
    plt.hist(cleaned, bins=30, density=True, color='gray', alpha=0.6, label="Predicted")
    # plt.plot(x, pdf_fitted, label=f"Best Fit: {best_fit}", linewidth=2, color="red")
    plt.xlabel("Value")
    plt.ylabel("Probability Density")
    plt.title(f"Probability Distribution for Samples of {attribute}")
    plt.legend()

In [ ]:
drawPlot(density_bnd, '평균수관밀도(%)')

# Test- Make CBH Raster

In [ ]:
result_dir = r"D:\ForestFire\CBH\result"
data_dir = r"D:\ForestFire\CBH\data"
params_file = r"D:\ForestFire\CBH\result\CR_Han_result_LogTransform_FIN2.csv"
# species_map, dbh_map, height_map, cr_map, dem, slope, azimuth
attribute_map_list = [os.path.join(result_dir, i) for i in ['KOFTR_GROU_raster.tif', 'DMCLS_CD_sampled.tif', 'HEIGHT_sampled.tif', 'DNST_CD_sampled.tif']] + \
[os.path.join(data_dir, i) for i in ['DEM_merge_5179.tif', 'Slope_rad_5179.tif', 'Aspect_merge_5179.tif']] # DMCLS_CD: 흉고직경, DNST_CD: 수관밀도
print(attribute_map_list)

In [ ]:
output_raster = createCBHRaster(params_file, attribute_map_list, result_dir, 5, 5)

### note

In [ ]:
def createSamples (data, attribute, size, bound = None):
    def _find_best_fit_distribution(filtered, distributions):
        results = {}
        x = np.linspace(filtered.min(), filtered.max(), 1000)
    
        for name, dist in distributions.items():
            try:
                # Fit distribution to data
                params = dist.fit(filtered)
                pdf_fitted = dist.pdf(x, *params)
                ks_stat, ks_pval = stats.kstest(filtered, lambda x: dist.cdf(x, *params))
    
                # Store results
                results[name] = {
                    "params": params,
                    "KS Statistic": ks_stat,
                    "p-value": ks_pval,
                    "pdf": pdf_fitted
                }
            except Exception as e:
                print(f"Skipping {name} due to error: {e}")
    
        # Select best fit (highest p-value)
        best_fit = max(results, key=lambda d: results[d]["p-value"])
        return best_fit, results
        
    distributions = {
    "Gamma": stats.gamma,
    "Log-Normal": stats.lognorm,
    "Beta": stats.beta,
    "Weibull": stats.weibull_min,
    "Exponential": stats.expon,
    "GEV" : stats.genextreme
    } 

    if bound != None:
        condition = (data[attribute] > bound[0]) & (data[attribute] <= bound[1])
        filtered_data = data.loc[condition, attribute].dropna()
        print('data filtered')
    else: filtered_data = data.loc[:, attribute].dropna()
    # q_low, q_high = filtered_data.quantile([0.01, 0.99])
    # filtered_data = filtered_data[(filtered_data >= q_low) & (filtered_data <= q_high)]
    best_fit, results = _find_best_fit_distribution(filtered_data, distributions)
    best_dist = distributions[best_fit]
    print(best_fit)
    samples = []
    while len(samples) < size:
        new_sample = best_dist.rvs(*results[best_fit]['params'], size=size, random_state=44)
        filtered_sample = new_sample[(new_sample > bound[0]) & (new_sample <= bound[1])]
        samples.extend(filtered_sample.tolist())
    samples = np.array(samples[:size])
    print(f'Created sample size: {len(samples)}')

    # plot the result of best_fit
    x = np.linspace(filtered_data.min(), filtered_data.max(), 1000)
    pdf_fitted = best_dist.pdf(x, *results[best_fit]['params'])
    
    plt.figure(figsize=(8, 5))
    plt.hist(filtered_data, bins=30, density=True, color='gray', alpha=0.6, label="Data Histogram")
    plt.plot(x, pdf_fitted, label=f"Best Fit: {best_fit}", linewidth=2, color="red")
    plt.xlabel(f"{attribute}")
    plt.ylabel("Probability Density")
    plt.title(f"Best-Fitting Probability Distribution for {attribute}")
    plt.legend()
    plt.show()
    
    # Print best fit results
    print(f"Best-Fitting Distribution: {best_fit}")
    print(f"Parameters: {results[best_fit]['params']}")
    print(f"KS Statistic: {results[best_fit]['KS Statistic']}")
    print(f"P-Value: {results[best_fit]['p-value']}")
    
    return samples, best_fit, results

In [ ]:
    # change string code to numerical code
    for target_feature in target_feature_lst:
        if target_feature == 'DMCLS_CD': # 흉고직경
            imsang[target_feature] = imsang[target_feature]].apply(lambda x : int(x) if x in ['1', '2', '3', '0'] else -99)
        elif target_feature == 'HEIGHT': # 수고
            imsang[target_feature] = imsang[target_feature].apply(lambda x : '16' if x == '15' else x).apply(lambda x : int(x) if x != ' ' else -99)
        elif target_feature == 'DNST_CD': # 수관밀도
            cd_code_dict = {'A' : 1, 'B' : 2, 'C' : 3}
            imsang[target_feature] = imsang[target_feature].apply(lambda x : cd_code_dict[x] if x in ['A', 'B', 'C'] else -99)
        elif target_feature == 'KOFTR_GROU':# 수종
            imsang[target_feature] = imsang[target_feature].astype('int')
        else:
            if imsang[[target_feature]].dtype() == 'object': print('Need to convert the data type to numerical one.')
            else: pass
"""# visuzlie new imsang raster
with rasterio.open(imsang_raster) as src:
    ras_data = src.read(2)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
plt.figure(figsize=(8, 6))
plt.imshow(ras_data, cmap='blue')
plt.colorbar(label='Species')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()"""

In [ ]:
## Create the distribution
# make height samples
alpha = 10.22 
beta = 43.14
loc = -1.56
scale = 76.72
h_samples = stats.beta.rvs(alpha, beta, loc=loc, scale=scale, size=100000)
h_samples = h_samples[h_samples > 0.]

# make dbh samples
shape = 1.58  # Shape parameter
loc = 7.56  # Location parameter
scale = 15.68   # Scale parameter
dbh_samples = stats.weibull_min.rvs(shape, loc=loc, scale=scale, size=100000)
dbh_samples = dbh_samples[dbh_samples > 0.]

# make cdown density samples
shape = 84.21  # Shape parameter
loc = -111.64  # Location parameter
scale = 2.31   # Scale parameter
cd_samples = stats.gamma.rvs(shape, loc=loc, scale=scale, size=10000)
cd_samples = cd_samples[(cd_samples > 0.) & (cd_samples <= 100.)]